In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
import time
from time import sleep
from contextlib import suppress


# TO DO
# Use match date to sort and filter duplicates. Also for history capping (1 year)

def setup_driver(headless = False):
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-notifications')
    if headless:
        options.add_argument('--headless')
    options.add_argument('--disable-gpu')  # Recommended for headless
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--no-sandbox')  # Bypass OS security model
    options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
    
    # Add a realistic user agent
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36')
    
    # Some additional useful options
    options.add_argument('--disable-blink-features=AutomationControlled')  # Hide automation
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # Hide automation 
    options.add_experimental_option('useAutomationExtension', False)  # Hide automation
    
    driver = webdriver.Chrome(options=options)
    
    # Execute JS to modify navigator.webdriver flag
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver



def get_league_name_and_country(header_text):
    """
    Extracts the league name from header text like 'USA : MLB Standings' or 'JAPAN : NPB Standings'
    Returns just the league name (e.g., 'MLB' or 'NPB')
    """
    try:
        parts = header_text.strip().split(':')[0]
        parts = parts.replace("\n", ":")
        if len(parts) > 1:
            country = parts.split(":")[1]
            league = parts.split(":")[0]
            return (league, country)
        return (header_text.strip(), "")
    except:
        return (header_text.strip(), "")
    

def get_exact(league_name):
    # Map of league names to their abbreviated codes
    if league_name == "MLB": return "MLB"
    if league_name == "MLB - Play Offs": return "MLB"
    if league_name == "NPB": return "NPB"
    if league_name == "NPB - Play Offs": return "NPB"
    if league_name == "KBO": return "KBO"
    if league_name == "KBO - Play Offs": return "KBO"
    if league_name == "CPBL": return "CPBL"
    if league_name == "CPBL - Play Offs": return "CPBL"
    if league_name == "LMB": return "LMB"
    if league_name == "LMB - Play Offs": return "LMB"
    if league_name == "LVBP": return "LVBP"
    if league_name == "LVBP - Play Offs": return "LVBP"
    if league_name == "ABL": return "ABL"
    if league_name == "ABL - Play Offs": return "ABL"
    if league_name == "Liga Mexicana del Pacífico": return "LMP"
    if league_name == "Liga Mexicana del Pacífico - Play Offs": return "LMP"
    if league_name == "Dominican Winter League": return "LIDOM"
    if league_name == "Dominican Winter League - Play Offs": return "LIDOM"
    if league_name == "MLB Spring Training": return "ST"
    if league_name == "NCAA": return "NCAA"
    if league_name == "NCAA - College World Series": return "NCAA"
    if league_name == "Minor League Baseball - AAA": return "AAA"
    if league_name == "Minor League Baseball - AA": return "AA"
    if league_name == "Minor League Baseball - A Advanced": return "A+"
    if league_name == "Minor League Baseball - A": return "A"
    return league_name


def is_desired_league(game_element):
    try:
        '''
        Starting from this game element, look backwards through the page until you find the first div that 
        has 'tournament__name' in its class name. Use this information to filter out absent leagues
        '''
        league_header = game_element.find_element(By.XPATH, "./preceding::div[contains(@class, 'headerLeague__wrapper')][1]")
        raw_text = league_header.text.strip()

        league_name, country = get_league_name_and_country(raw_text)
        league_name = league_name.replace('\n', '')
        league_name = league_name.replace('Draw', '')

        desired_leagues = [
                            'MLB' # Major League Baseball - USA
                            # 'MLB - Play Offs',
                            # 'NPB', # Nippon Professional Baseball - Japan
                            # 'NPB - Play Offs',
                            # 'KBO', # Korean Baseball Organization - South Korea
                            # 'KBO - Play Offs',
                            # 'CPBL', # Chinese Professional Baseball League - Taiwan
                            # 'CPBL - Play Offs',
                            # 'LMB', # Mexican Baseball League
                            # 'LMB - Play Offs',
                            # 'LVBP', # Venezuelan Professional Baseball League
                            # 'LVBP - Play Offs',
                            # 'ABL', # Australian Baseball League
                            # 'ABL - Play Offs',
                            # 'Liga Mexicana del Pacífico', # Mexican Pacific League
                            # 'Liga Mexicana del Pacífico - Play Offs',
                            # 'Dominican Winter League',
                            # 'Dominican Winter League - Play Offs',
                            # 'MLB Spring Training',
                            # 'NCAA',
                            # 'NCAA - College World Series',
                            # 'Minor League Baseball - AAA',
                            # 'Minor League Baseball - AA',
                            # 'Minor League Baseball - A Advanced',
                            # 'Minor League Baseball - A',
                        ]
        return (any(league == league_name for league in desired_leagues), get_exact(league_name), country)
                
    except NoSuchElementException:
        return (False, "", "")
    

def get_upcoming_games(driver, day = 0):
    driver.get("https://www.flashscore.com/baseball/")

    with suppress(Exception):
        accept_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        accept_button.click()
    
    upcoming = []
    if day > 0:
        for _ in range(day):
            next = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-day-picker-arrow='next']"))
            )
            next = driver.find_element(By.CSS_SELECTOR, "button[data-day-picker-arrow='next']")
            driver.execute_script("arguments[0].click();", next)
            sleep(3)


    try:
        # Wait for games to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "event__match"))
        )
        games = driver.find_elements(By.CLASS_NAME, "event__match")

        for game in games:
            try:
                is_league, league, country = is_desired_league(game)
                if is_league:
                    teams = game.find_elements(By.CLASS_NAME, "event__participant")
                    time = game.find_element(By.CLASS_NAME, "event__time")
                    game_link = game.find_element(By.CLASS_NAME, "eventRowLink").get_attribute("href")
                    
                    upcoming.append({
                        'league': league,
                        'country': country,
                        'home': teams[0].text,
                        'away': teams[1].text,
                        'time': time.text[:5],
                        'link': game_link
                    })
            except Exception as e:
                continue
    except Exception as e:
        print(f"Error getting upcoming games: {e}")

    return upcoming


def get_team_last_matches(driver, element, target_league, section_index):
    target_league = target_league.lower()
    matches = []
    
    length = 4 if section_index < 2 else 2

    for _ in range(length):
        try:
            show_more_button = element.find_element(By.CLASS_NAME, "wclButtonLink--h2h")
            time.sleep(1)
            driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
            driver.execute_script("arguments[0].click();", show_more_button)
            time.sleep(2)
        except Exception as e:
            print(f'Error clicking show more icon: {e}')

    try:
        rows = element.find_elements(By.CLASS_NAME, "h2h__row")

        # For baseball, keep a one-year history for most leagues
        cutoff_date = datetime.now() - timedelta(days=365)

        for row in rows:
            try:
                date_str = row.find_element(By.CLASS_NAME, "wclH2h__date").text
                match_date = datetime.strptime(date_str, '%d.%m.%y')

                league = row.find_element(By.CLASS_NAME, "h2h__event").text
                league = league.lower()

                if target_league.startswith(league) and match_date > cutoff_date:
                    home_team = row.find_element(By.CLASS_NAME, "h2h__homeParticipant").text
                    away_team = row.find_element(By.CLASS_NAME, "h2h__awayParticipant").text
                    score = row.find_element(By.CLASS_NAME, "h2h__result").text

                    match_data = {
                        'date': date_str,
                        'home': home_team,
                        'away': away_team,
                        'score': score,
                        'league': league
                    }
                    
                    matches.append(match_data)
            except Exception as e:
                print(f"Error processing match row: {e}")
                continue
                
    except Exception as e:
        print(f"Error getting matches: {e}")
    
    # Limit the number of matches
    matches = matches[:15] if section_index < 2 else matches[:5]
    
    return matches

def scrape_h2h_page(driver, url, league):
    try:
        driver.get(url)
        # Handle cookie consent if present
        with suppress(Exception):
            accept_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            accept_button.click()
            
        # Click H2H tab and wait for it to load
        try:
            sleep(5)
            tab_buttons = driver.find_elements(By.CSS_SELECTOR, "div.detailOver > div > a")
            h2h_index = 0
            for tab_button in tab_buttons:
                tab_text:str = tab_button.text
                if tab_text.startswith("H2H"): break
                h2h_index += 1
            h2h_button = tab_buttons[h2h_index]
            driver.execute_script("arguments[0].click();", h2h_button)
            time.sleep(2)  # Wait for tab to load
        except Exception as e:
            print(f"Error clicking H2H tab: {e}")

        # Get sections with explicit wait
        sections = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "h2h__section"))
        )
        
        results = {
            'home_matches': get_team_last_matches(driver, sections[0], league, 0),
            'away_matches': get_team_last_matches(driver, sections[1], league, 1),
            'h2h_matches': get_team_last_matches(driver, sections[2], league, 2)
        }
        
        return results
        
    except Exception as e:
        print(f"Error in scrape_h2h_page: {e}")
        return {'home_matches': [], 'away_matches': [], 'h2h_matches': []}


def main():
    # 0 for today, 1 for next day games
    day = 1
    
    driver = setup_driver(headless=True)
    try:
        # Get today's upcoming games
        upcoming = get_upcoming_games(driver, day)

        number_of_games = len(upcoming)

        file1 = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Baseball\MLB1.txt"
        # file2 = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Baseball\random1.txt"

        # file1_countries = ["USA"]  # MLB games go to file1
        # All other countries go to file2 (Japan, Korea, Taiwan, Mexico, Venezuela, etc.)

        # For each upcoming game, get last 15 scores and H2H
        last_saved = 0  # default value should be 0
        for number, game in enumerate(upcoming):
            if (number+1) > last_saved:
                print(f'{number+1}/{number_of_games}', '\r', end = '')
                
                home_team = game['home']
                away_team = game['away']
                country   = game['country']
                league    = game['league']
                game_time = game['time']

                home_score = []
                away_score = []

                results = scrape_h2h_page(driver, game['link'], league)

                # Process home team's recent games
                for match in results['home_matches']:
                    score = match['score'].strip().split()
                    if home_team == match['home']:
                        home_score.append(score[0])
                    else:
                        home_score.append(score[1])

                # Process away team's recent games
                for match in results['away_matches']:
                    score = match['score'].strip().split()
                    if away_team == match['home']:
                        away_score.append(score[0])
                    else:
                        away_score.append(score[1])

                # Add separator and process head-to-head games
                home_score.append(':')
                away_score.append(':')
                for match in results['h2h_matches']:
                    score = match['score'].strip().split()
                    if home_team == match['home']:
                        home_score.append(score[0])
                        away_score.append(score[1]) 
                    else:
                        home_score.append(score[1])
                        away_score.append(score[0])

                # Choose which file to write to based on country
                file = file1 
                # if country in file1_countries else file2

                with open(file, 'a') as fileObj:
                    fileObj.write(f'{home_team}: ')
                    fileObj.write(' '.join(str(num) for num in home_score))
                    fileObj.write('\n')
                    fileObj.write(f'{away_team}: ')
                    fileObj.write(' '.join(str(num) for num in away_score))
                    fileObj.write('\n')
                    fileObj.write(f'({country}, {league}, {game_time})\n\n')
                time.sleep(3)   
    except Exception as e:
        print(f"Error in main: {e}")
    finally:
        driver.quit()

if __name__ == "__main__":
    main()